In [1]:
!pip -q install -U nvidia-ml-py3 psutil pandas bitsandbytes accelerate peft transformers

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.3/263.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
import os, time, json
import pandas as pd
import torch
import torch.nn as nn
import psutil

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from pynvml import (
    nvmlInit, nvmlShutdown, nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetPowerUsage
)

assert torch.cuda.is_available(), "GPU not detected. In Colab: Runtime > Change runtime type > GPU."
device = "cuda"
proc = psutil.Process()

print("Torch:", torch.__version__)

Torch: 2.9.0+cu126


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PHASE2_ADAPTER_PATH = "/content/drive/My Drive/ML_Project_Phase2/phi3_squad_final"
PHASE3_HEAD_PATH    = "/content/drive/My Drive/ML_Project_Phase3/early_exit_head_block9.pt"
BASE_MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

MAX_NEW_TOKENS = 10
MAX_INPUT_TOKENS = 128

# Early-exit settings (block 9 -> 0-based index 8)
EXIT_BLOCK_INDEX = 8
EE_THRESHOLD = 0.8

PROMPT = "<|user|>\nContext: Toronto is a city in Canada.\nQuestion: What country is Toronto in?\n<|assistant|>\n"

RUN_ALL = True
MODE = "pretrained"  # used only if RUN_ALL=False. One of: pretrained, finetuned, early_exit

In [5]:
OUT_DIR = "/content/drive/My Drive/ML_Project_Phase4_Outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

OUT_DIR: /content/drive/My Drive/ML_Project_Phase4_Outputs


In [6]:
# 4-bit load if bitsandbytes is available
USE_4BIT = True
try:
    import bitsandbytes as bnb  # noqa
except Exception:
    USE_4BIT = False

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    print("Using 4-bit quantization (bnb).")
else:
    print("bitsandbytes not available -> using FP16 (may OOM on T4).")

# Tokenizer (use adapter path so it matches your finetuned prompting)
tokenizer = AutoTokenizer.from_pretrained(PHASE2_ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model (Phase 1)
if USE_4BIT:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
    )

# Finetuned model wrapper (Phase 2 / Phase 3)
ft_model = PeftModel.from_pretrained(base_model, PHASE2_ADAPTER_PATH)

base_model.eval()
ft_model.eval()
for p in base_model.parameters():
    p.requires_grad = False
for p in ft_model.parameters():
    p.requires_grad = False

print("Loaded base + finetuned models.")

Using 4-bit quantization (bnb).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Loaded base + finetuned models.


In [7]:
# Early-exit head (Phase 3)
class EarlyExitHead(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, vocab_size),
        )
    def forward(self, hidden_states):
        return self.ff(self.ln(hidden_states))

def load_ee_head(path: str):
    ckpt = torch.load(path, map_location="cpu")
    head = EarlyExitHead(ckpt["hidden_size"], ckpt["vocab_size"]).to(device)
    head.load_state_dict(ckpt["head_state_dict"])
    head.eval()
    return head, ckpt

ee_head, ee_ckpt = load_ee_head(PHASE3_HEAD_PATH)
print("Loaded early-exit head. hidden_size=", ee_ckpt["hidden_size"], "vocab=", ee_ckpt["vocab_size"])

Loaded early-exit head. hidden_size= 3072 vocab= 32064


In [8]:
def unwrap_model(m):
    if hasattr(m, "base_model"):
        m = m.base_model
    if hasattr(m, "model"):
        m = m.model
    return m

def get_blocks(m):
    cands = [m, unwrap_model(m)]
    for c in cands:
        if hasattr(c, "model") and hasattr(c.model, "layers"):
            return c.model.layers
        if hasattr(c, "layers"):
            return c.layers
        if hasattr(c, "transformer") and hasattr(c.transformer, "h"):
            return c.transformer.h
        if hasattr(c, "transformer") and hasattr(c.transformer, "layers"):
            return c.transformer.layers
    raise ValueError("Could not locate transformer blocks.")

def forward_until_block_hidden(model, input_ids, attention_mask, block_index: int):
    blocks = get_blocks(model)
    assert 0 <= block_index < len(blocks)

    class _Grab(Exception):
        def __init__(self, hs): self.hs = hs

    def hook_fn(mod, inputs, output):
        hs = output[0] if isinstance(output, (tuple, list)) else output
        raise _Grab(hs)

    h = blocks[block_index].register_forward_hook(hook_fn)
    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, output_hidden_states=False)
    except _Grab as e:
        return e.hs
    finally:
        h.remove()
    raise RuntimeError("Hidden state not captured.")

def greedy_next_token_from_logits(logits):
    probs = torch.softmax(logits, dim=-1)
    conf, pred = probs.max(dim=-1)
    return pred, float(conf.item())

# End token ids (stop generation when produced)
END_IDS = set()
if tokenizer.eos_token_id is not None:
    END_IDS.add(int(tokenizer.eos_token_id))
for t in ["<|end|>", "<|endoftext|>"]:
    tid = tokenizer.convert_tokens_to_ids(t)
    if tid is not None and tid != tokenizer.unk_token_id:
        END_IDS.add(int(tid))

print("END_IDS:", END_IDS)

END_IDS: {32000, 32007}


In [9]:
def _nvml_energy_mj(handle):
    from pynvml import nvmlDeviceGetTotalEnergyConsumption
    return nvmlDeviceGetTotalEnergyConsumption(handle)

def _nvml_power_w(handle):
    return nvmlDeviceGetPowerUsage(handle) / 1000.0

class StopAfterBlock(Exception):
    pass

def profile_one_forward_blocks(model, input_ids, attention_mask, stop_after_block_index=None):
    blocks = get_blocks(model)
    stats = {}

    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)

    energy_counter_ok = True
    try:
        _ = _nvml_energy_mj(handle)
    except Exception:
        energy_counter_ok = False

    handles = []

    def make_hooks(layer_idx):
        state = {}
        def pre_hook(mod, inputs):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            state["t0"] = time.perf_counter()
            state["rss0_mb"] = proc.memory_info().rss / (1024**2)
            if energy_counter_ok:
                state["e0_mj"] = _nvml_energy_mj(handle)
            else:
                state["p0_w"] = _nvml_power_w(handle)

        def post_hook(mod, inputs, output):
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            dt = t1 - state["t0"]

            rss1_mb = proc.memory_info().rss / (1024**2)
            cpu_rss_mb = max(state["rss0_mb"], rss1_mb)

            gpu_peak_alloc_mb = torch.cuda.max_memory_allocated() / (1024**2)

            if energy_counter_ok:
                e1_mj = _nvml_energy_mj(handle)
                energy_j = max(0.0, (e1_mj - state["e0_mj"]) / 1000.0)
            else:
                p1_w = _nvml_power_w(handle)
                energy_j = max(0.0, 0.5 * (state.get("p0_w", p1_w) + p1_w) * dt)

            stats[layer_idx] = {
                "time_ms": dt * 1000.0,
                "energy_j": float(energy_j),
                "gpu_peak_alloc_MB": float(gpu_peak_alloc_mb),
                "cpu_rss_MB": float(cpu_rss_mb),
            }

        return pre_hook, post_hook

    for i, blk in enumerate(blocks):
        pre, post = make_hooks(i)
        handles.append(blk.register_forward_pre_hook(pre))
        handles.append(blk.register_forward_hook(post))

    stop_handle = None
    if stop_after_block_index is not None:
        blocks = get_blocks(model)
        def stop_hook(mod, inputs, output):
            raise StopAfterBlock()
        stop_handle = blocks[stop_after_block_index].register_forward_hook(stop_hook)

    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, output_hidden_states=False)
    except StopAfterBlock:
        pass
    finally:
        for h in handles:
            h.remove()
        if stop_handle is not None:
            stop_handle.remove()
        nvmlShutdown()

    return stats, energy_counter_ok

In [10]:
def run_mode(mode: str):
    assert mode in {"pretrained", "finetuned", "early_exit"}
    model = base_model if mode == "pretrained" else ft_model

    enc = tokenizer(PROMPT, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    # Warmup
    with torch.inference_mode():
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)

    n_layers = len(get_blocks(model))
    agg = [{
        "layer": i,
        "executed_steps": 0,
        "total_time_ms": 0.0,
        "total_energy_j": 0.0,
        "gpu_peak_alloc_MB": 0.0,
        "cpu_rss_MB": 0.0,
    } for i in range(n_layers)]

    steps_exited = 0
    steps_full = 0
    energy_counter_ok_final = None
    step_log = []

    for step in range(MAX_NEW_TOKENS):
        stop_after = None
        exited = False
        conf = None
        next_tok = None

        if mode == "early_exit":
            hs = forward_until_block_hidden(ft_model, input_ids, attention_mask, block_index=EXIT_BLOCK_INDEX)
            hs_dtype = hs.dtype
            head = ee_head.to(device=device, dtype=hs_dtype)

            with torch.no_grad():
                last_h = hs[:, -1, :].clone().to(dtype=hs_dtype)
                ee_logits = head(last_h)
                next_tok_ee, conf = greedy_next_token_from_logits(ee_logits)

            if conf >= EE_THRESHOLD:
                stop_after = EXIT_BLOCK_INDEX
                exited = True
                steps_exited += 1
                next_tok = next_tok_ee
            else:
                steps_full += 1

        layer_stats, energy_counter_ok = profile_one_forward_blocks(
            model, input_ids, attention_mask, stop_after_block_index=stop_after
        )
        energy_counter_ok_final = energy_counter_ok_final if energy_counter_ok_final is not None else energy_counter_ok

        if not exited:
            with torch.no_grad():
                out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
                last_logits = out.logits[:, -1, :]
                next_tok, conf2 = greedy_next_token_from_logits(last_logits)
                if mode != "early_exit":
                    conf = conf2

        for i in range(n_layers):
            s = layer_stats.get(i)
            if s is None:
                continue
            agg[i]["executed_steps"] += 1
            agg[i]["total_time_ms"] += float(s["time_ms"])
            agg[i]["total_energy_j"] += float(s["energy_j"])
            agg[i]["gpu_peak_alloc_MB"] = max(agg[i]["gpu_peak_alloc_MB"], float(s["gpu_peak_alloc_MB"]))
            agg[i]["cpu_rss_MB"] = max(agg[i]["cpu_rss_MB"], float(s["cpu_rss_MB"]))

        step_log.append({
            "step": step,
            "exited_early": bool(exited),
            "confidence": float(conf) if conf is not None else None,
            "input_len": int(input_ids.shape[1]),
            "next_token_id": int(next_tok.item()),
            "next_token_text": tokenizer.decode(next_tok),
        })

        # append token
        input_ids = torch.cat([input_ids, next_tok.view(1, 1)], dim=1)
        if attention_mask is not None:
            attention_mask = torch.cat([attention_mask, torch.ones((1,1), device=device, dtype=attention_mask.dtype)], dim=1)

        # stop on end token
        if int(next_tok.item()) in END_IDS:
            break

    if mode != "early_exit":
        steps_exited = 0
        steps_full = len(step_log)

    df_layers = pd.DataFrame(agg)
    df_steps = pd.DataFrame(step_log)

    meta = {
        "energy_counter_ok": bool(energy_counter_ok_final),
        "mode": mode,
        "max_new_tokens": int(MAX_NEW_TOKENS),
        "steps_exited": int(steps_exited),
        "steps_full": int(steps_full),
        "exit_block_index": int(EXIT_BLOCK_INDEX),
        "threshold": float(EE_THRESHOLD),
        "prompt": PROMPT,
    }
    return df_layers, df_steps, meta

In [14]:
modes = ["pretrained", "finetuned", "early_exit"] if RUN_ALL else [MODE]

all_layers = {}
all_steps = {}
all_meta = {}

for md in modes:
    print("\n" + "="*90)
    print("Running:", md)
    df_layers, df_steps, meta = run_mode(md)

    all_layers[md] = df_layers
    all_steps[md] = df_steps
    all_meta[md] = meta

    print("Meta:", meta)
    if md == "early_exit":
        print("Early-exit steps:", meta["steps_exited"], "| Full steps:", meta["steps_full"])
        display(df_steps)  # <-- token table

    display(df_layers)

summary_rows = []
for md in modes:
    df = all_layers[md]
    meta = all_meta[md]
    summary_rows.append({
        "mode": md,
        "total_energy_j": float(df["total_energy_j"].sum()),
        "total_time_ms": float(df["total_time_ms"].sum()),
        "max_gpu_peak_alloc_MB": float(df["gpu_peak_alloc_MB"].max()),
        "max_cpu_rss_MB": float(df["cpu_rss_MB"].max()),
        "steps_exited": int(meta.get("steps_exited", 0)),
        "steps_full": int(meta.get("steps_full", 0)),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df


Running: pretrained
Meta: {'energy_counter_ok': False, 'mode': 'pretrained', 'max_new_tokens': 10, 'steps_exited': 0, 'steps_full': 2, 'exit_block_index': 8, 'threshold': 0.8, 'prompt': '<|user|>\nContext: Toronto is a city in Canada.\nQuestion: What country is Toronto in?\n<|assistant|>\n'}


,layer,executed_steps,total_time_ms,total_energy_j,gpu_peak_alloc_MB,cpu_rss_MB
0,0,2,16.534674,0.848558,2576.229004,2966.230469
1,1,2,15.351134,0.731936,2576.357910,2966.230469
2,2,2,14.226795,0.692689,2576.357910,2966.230469
3,3,2,13.565967,0.621033,2576.357910,2966.230469
4,4,2,14.601191,0.631705,2576.357910,2966.230469
5,5,2,14.701605,0.636193,2576.357910,2966.230469
6,6,2,15.143524,0.655089,2576.357910,2966.230469
7,7,2,13.677412,0.590641,2576.357910,2966.230469
8,8,2,13.477162,0.582198,2576.357910,2966.230469
9,9,2,13.495000,0.582978,2576.357910,2966.230469



Running: finetuned
Meta: {'energy_counter_ok': False, 'mode': 'finetuned', 'max_new_tokens': 10, 'steps_exited': 0, 'steps_full': 2, 'exit_block_index': 8, 'threshold': 0.8, 'prompt': '<|user|>\nContext: Toronto is a city in Canada.\nQuestion: What country is Toronto in?\n<|assistant|>\n'}


,layer,executed_steps,total_time_ms,total_energy_j,gpu_peak_alloc_MB,cpu_rss_MB
0,0,2,14.824657,1.011080,2576.229004,2966.230469
1,1,2,13.136592,0.881884,2576.357910,2966.230469
2,2,2,15.150475,0.955327,2576.357910,2966.230469
3,3,2,13.861793,0.825404,2576.357910,2966.230469
4,4,2,14.655527,0.874081,2576.357910,2966.230469
5,5,2,19.258722,1.156813,2576.357910,2966.230469
6,6,2,13.109364,0.779167,2576.357910,2966.230469
7,7,2,13.417708,0.797447,2576.357910,2966.230469
8,8,2,15.936454,0.941940,2576.357910,2966.230469
9,9,2,13.127422,0.754775,2576.357910,2966.230469



Running: early_exit
Meta: {'energy_counter_ok': False, 'mode': 'early_exit', 'max_new_tokens': 10, 'steps_exited': 1, 'steps_full': 1, 'exit_block_index': 8, 'threshold': 0.8, 'prompt': '<|user|>\nContext: Toronto is a city in Canada.\nQuestion: What country is Toronto in?\n<|assistant|>\n'}
Early-exit steps: 1 | Full steps: 1


,step,exited_early,confidence,input_len,next_token_id,next_token_text
0,0,False,0.112915,21,7400,Canada
1,1,True,0.992676,22,32007,<|end|>


,layer,executed_steps,total_time_ms,total_energy_j,gpu_peak_alloc_MB,cpu_rss_MB
0,0,2,14.744172,1.049069,2576.425781,2966.230469
1,1,2,14.795710,1.046875,2576.554688,2966.230469
2,2,2,20.429234,1.465465,2576.554688,2966.230469
3,3,2,13.168538,0.934842,2576.554688,2966.230469
4,4,2,13.642981,0.882355,2576.554688,2966.230469
5,5,2,13.144251,0.767970,2576.554688,2966.230469
6,6,2,13.427768,0.719445,2576.554688,2966.230469
7,7,2,13.318268,0.644442,2576.554688,2966.230469
8,8,2,13.216181,0.639490,2576.554688,2966.230469
9,9,1,6.626701,0.322402,2573.519531,2966.230469


,mode,total_energy_j,total_time_ms,max_gpu_peak_alloc_MB,max_cpu_rss_MB,steps_exited,steps_full
0,pretrained,19.658234,449.419866,2576.357910,2966.230469,0,2
1,finetuned,26.139213,456.862576,2576.357910,2966.230469,0,2
2,early_exit,16.988513,307.752012,2576.554688,2966.230469,1,1


In [12]:
for md in modes:
    all_layers[md].to_csv(os.path.join(OUT_DIR, f"{md}_layer_table.csv"), index=False)
    all_steps[md].to_csv(os.path.join(OUT_DIR, f"{md}_token_table.csv"), index=False)
    with open(os.path.join(OUT_DIR, f"{md}_meta.json"), "w") as f:
        json.dump(all_meta[md], f, indent=2)

summary_df.to_csv(os.path.join(OUT_DIR, "summary_compare.csv"), index=False)
print("Saved to:", OUT_DIR)

Saved to: /content/drive/My Drive/ML_Project_Phase4_Outputs
